# Lounge scorecard: site demand + site rent

**In one line:** each lounge's SITE is scored on two signals, both ranked within its emirate's frozen market of competitor salon sites: demand (`demand_pct`, from `site_served`) and rent (`rent_ratio`). Equal weights; composite ≥ 0.65 is PROTECT, ≤ 0.35 SHRINK, HOLD between. The lounge's own performance (`performance_vs_site`), the UAE-wide percentiles and cannibalisation (`absorbed_share`) are context, not scored.

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women, Huff** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = shop rent at the site** (`src/features/lounges.py`). Shop rent per cell from three sources (2026-10-11): DLD Ejari contracts (Dubai), ADREC 2025 district rates (Abu Dhabi), shop listings' asking rents (northern emirates), else the emirate median. The cost signal is `rent_ratio` = the emirate's median shop rent at its reference salon sites ÷ the site's rent (> 1 = cheaper than the typical salon site in its emirate), scored on a frozen scale: `RENT_RATIO_P10` = 0.6176 → 0, `RENT_RATIO_P90` = 1.4290 → 1, so a ratio of 1 scores 0.47 (`RENT_RATIO_WHY`). Rent only: served women are the demand signal and aren't counted twice. **Fallback rents are neutral** (`growth.RENT_FALLBACK_WHY`): a cell with no DLD, ADREC or listings rent takes its emirate's median, which is no observation of the site, so a lounge whose site cell is on it scores rent 0.5 and an area with half or more of its rented women on it passes the rent test; both calls are low confidence. `site_cost_k` (rent × 150 m² + an assumed fit-out over 5 years) is shown, not scored.
3. **Lounge scorecard** = site demand + site rent, both judged **within the emirate** against a frozen reference of 1,123 real premium competitor salon sites (no imputed, no Bedashing; `data/seed/v3/site_reference.csv`, `SITE_REFERENCE_WHY`): Dubai 582, Abu Dhabi 304, Sharjah 143, Ajman 38, Ras Al Khaimah 33, Fujairah 18, Umm Al Quwain 5. Never against our own lounges, so opening or closing one moves nothing. Site demand (`site_served`) = the women a typical lounge (frozen at 757 reviews and 4.6★, the 2026-10-11 median of our open scored lounges: `TYPICAL_WHY`) would serve at this lounge's site, with this lounge closed and its siblings and rivals open; it is scored as `demand_pct`, the share of the emirate's reference sites it out-serves (UAE-wide for the four emirates under 50 sites; `demand_ref` says which). `demand_pct_uae` and `rent_pct_uae` give the UAE-wide percentiles as context, not scored. The lounge's own served women are context too: `performance_vs_site` = served ÷ site-served ("The model estimates this lounge is performing 2.7x compared to its site"). Cannibalisation (`absorbed_share`, the share of a lounge's women its siblings would absorb if it closed) is shown, not scored. PROTECT at a composite ≥ 0.65, SHRINK ≤ 0.35, HOLD between.
4. **Growth** = what-if: open the same frozen typical lounge at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new. GROW needs net new ≥ 6,000, worker housing under 50%, `rent_ratio` ≥ 1 (unknown, or mostly the emirate-median fallback, passes at low confidence) and `demand_pct` ≥ 0.5, all against the same salon-site reference.

5. **Dropped:** the UAE-wide cost anchor (`COST_ANCHOR`, a cost index of site-served women per AED 1k: it marked down every dear Dubai site and counted served women twice), the emirate-median-of-lounges baseline (it moved when a sibling opened or closed), capture and `est_customers` as scored signals (Huff already splits by reviews), the JLL mall-class rent bands (superseded by ADREC), and affluence weighting (off: `features/affluence.ipynb`).

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Signals and the salon-site reference

In [2]:
pd.set_option("display.max_colwidth", 200)
print(L.SITE_REFERENCE_WHY, "\n")
print(scorecard.RENT_RATIO_WHY, "\n")
ref = L.site_reference()
print("reference sites:", len(ref), "| MIN_REF_SITES", L.MIN_REF_SITES, "| RENT_RATIO_P10", scorecard.RENT_RATIO_P10, "| RENT_RATIO_P90", scorecard.RENT_RATIO_P90)
display(ref.groupby("emirate").agg(sites=("place_id", "size"), ref_served_p25=("ref_served", lambda x: x.quantile(.25)),
        ref_served_median=("ref_served", "median"), ref_served_p75=("ref_served", lambda x: x.quantile(.75)),
        rent_median=("shop_rent_m2", "median")).assign(demand_ranked=lambda x: np.where(x.sites >= L.MIN_REF_SITES, "own emirate", "UAE-wide"))
        .sort_values("sites", ascending=False).round(0))
sc = F[~F.not_scored]
print("median scored demand_pct:", round(sc.demand_pct.median(), 3), "| median scored rent_ratio:", round(sc.rent_ratio.median(), 3))
pd.DataFrame([dict(signal=s.name, field=s.field, worst=s.worst, best=s.best, weight=s.weight, why=s.why) for s in scorecard.SIGNALS])

Labels are judged within each emirate, against its market of salon SITES, frozen: every real (non-imputed) premium competitor salon in the served model, Bedashing excluded (1,123 sites; data/seed/v3/site_reference.csv, built once 2026-10-11 by scripts/build_site_reference.py). Per site: the cell shop rent, and the women a typical lounge would serve there with all our lounges open. Never against our own lounges (Fujairah and Ras Al Khaimah have one, Sharjah two), and nothing we open or close moves it. Demand = the share of the emirate's reference sites a site out-serves; an emirate with under 50 reference sites (Ajman 38, Ras Al Khaimah 33, Fujairah 18, Umm Al Quwain 5) uses the UAE-wide set instead: a percentile among a few dozen sites moves several points per site. Rent ratio = the emirate's median shop rent at its reference sites ÷ the site's shop rent (its own emirate's median even under 50 sites: rents are an emirate-level market, and a UAE-wide median would call every northern sit

,sites,ref_served_p25,ref_served_median,ref_served_p75,rent_median,demand_ranked
emirate,,,,,,
Dubai,582,1815.0,2328.0,2926.0,2154.0,own emirate
Abu Dhabi,304,1448.0,3950.0,6366.0,1394.0,own emirate
Sharjah,143,3490.0,5905.0,8474.0,1121.0,own emirate
Ajman,38,9496.0,11100.0,12130.0,753.0,UAE-wide
Ras Al Khaimah,33,6668.0,7102.0,8676.0,853.0,UAE-wide
Fujairah,18,6940.0,7964.0,8225.0,745.0,UAE-wide
Umm Al Quwain,5,7226.0,10977.0,11424.0,543.0,UAE-wide


median scored demand_pct: 0.657 | median scored rent_ratio: 0.904


,signal,field,worst,best,weight,why
0,demand,demand_pct,0.0000,1.000,1.0,"Scores the SITE, not how much the lounge itself sells: the women a typical lounge (median reviews and rating) would serve there, this lounge closed and rivals and siblings open, ranked among the f..."
1,cost,rent_ratio,0.6176,1.429,1.0,"Above 1 = cheaper than a typical salon site in its emirate. Scale frozen (RENT_RATIO_WHY). Rent only: served women are already the demand signal, so they are not counted twice. Site shop rent from..."


## 2. The calls, with reasons

`reason` gives each signal's value and score. A site cell on the emirate-median fallback (`site_rent_source` = `emirate_median`) scores cost a neutral 0.5 whatever its ratio says.

In [3]:
rows = []
for b, d in D.items():
    if d.action == "NOT SCORED": continue
    x = F.loc[b]
    fb = growth.rent_fallback(r.features[list(F.index).index(b)])
    reason = (f"site serves {x.site_served:,.0f} (demand {d.scores['demand']:.2f}); "
              + (f"rent ratio {x.rent_ratio:.2f} is the {x.emirate} fallback median (cost neutral {d.scores['cost']:.2f})" if fb
                 else f"rent ratio {x.rent_ratio:.2f} (cost {d.scores['cost']:.2f})"))
    rows.append(dict(lounge=b, emirate=x.emirate, call=d.action, composite=round(d.composite, 3),
                     site_served=round(x.site_served), demand_ref=x.demand_ref, demand_pct=round(x.demand_pct, 2), demand_pct_uae=round(x.demand_pct_uae, 2),
                     rent_ratio=round(x.rent_ratio, 2), rent_source=x.site_rent_source, rent_pct_uae=round(x.rent_pct_uae, 2),
                     served=round(x.served_women), perf_vs_site_context=round(x.performance_vs_site, 1),
                     confidence=d.confidence, reason=reason, weight_flips="; ".join(d.weight_flips)))
t = pd.DataFrame(rows).sort_values("composite", ascending=False).set_index("lounge")
print(t.call.value_counts().to_dict(), "| not scored:", [d.branch_id for d in r.decisions if d.action == "NOT SCORED"])
print(pd.crosstab(t.emirate, t.call))
with pd.option_context("display.max_colwidth", 90): display(t)

{'HOLD': 14, 'PROTECT': 6, 'SHRINK': 3} | not scored: ['zayed-international-airport']
call            HOLD  PROTECT  SHRINK
emirate                              
Abu Dhabi         10        3       1
Dubai              2        2       1
Fujairah           0        1       0
Ras Al Khaimah     1        0       0
Sharjah            1        0       1


,emirate,call,composite,site_served,demand_ref,demand_pct,demand_pct_uae,rent_ratio,rent_source,rent_pct_uae,served,perf_vs_site_context,confidence,reason,weight_flips
lounge,,,,,,,,,,,,,,,
al-taif-mall,Fujairah,PROTECT,0.950,8980,UAE,0.90,0.90,1.55,listings,0.00,6828,0.8,high,"site serves 8,980 (demand 0.90); rent ratio 1.55 (cost 1.00)",
mirdif-35,Dubai,PROTECT,0.790,5388,Dubai,0.98,0.75,1.10,dld,0.58,5214,1.0,high,"site serves 5,388 (demand 0.98); rent ratio 1.10 (cost 0.59)",
al-dhafra,Abu Dhabi,PROTECT,0.761,22852,Abu Dhabi,1.00,1.00,1.04,adrec,0.32,17634,0.8,high,"site serves 22,852 (demand 1.00); rent ratio 1.04 (cost 0.52)",
al-ain,Abu Dhabi,PROTECT,0.724,10173,Abu Dhabi,0.95,0.93,1.19,emirate_median,0.22,18652,1.8,low,"site serves 10,173 (demand 0.95); rent ratio 1.19 is the Abu Dhabi fallback median (co...",
nad-al-sheba,Dubai,PROTECT,0.705,2670,Dubai,0.68,0.48,1.21,dld,0.55,2229,0.8,medium,"site serves 2,670 (demand 0.68); rent ratio 1.21 (cost 0.73)",
al-falah,Abu Dhabi,PROTECT,0.664,7866,Abu Dhabi,0.83,0.85,1.19,emirate_median,0.22,9510,1.2,low,"site serves 7,866 (demand 0.83); rent ratio 1.19 is the Abu Dhabi fallback median (cos...",demand weight ×0.75 → HOLD; cost weight ×1.25 → HOLD
mohammed-bin-zayed-city,Abu Dhabi,HOLD,0.607,4581,Abu Dhabi,0.59,0.70,1.13,adrec,0.26,4057,0.9,low,"site serves 4,581 (demand 0.59); rent ratio 1.13 (cost 0.63)",
shahama,Abu Dhabi,HOLD,0.575,9865,Abu Dhabi,0.93,0.92,0.80,adrec,0.54,9865,1.0,medium,"site serves 9,865 (demand 0.93); rent ratio 0.80 (cost 0.22)",
baniyas,Abu Dhabi,HOLD,0.543,13036,Abu Dhabi,0.99,0.98,0.70,adrec,0.62,11086,0.9,high,"site serves 13,036 (demand 0.99); rent ratio 0.70 (cost 0.10)",


In [4]:
# The fallback-rent rule (growth.RENT_FALLBACK_WHY): what the emirate-median ratio would have scored
cost = next(s for s in scorecard.SIGNALS if s.name == "cost")
fb = t[t.rent_source == "emirate_median"]
print(growth.RENT_FALLBACK_WHY, "\n")
fb.assign(cost_if_ratio_scored=[round(scorecard.score(cost, x), 2) for x in fb.rent_ratio],
          composite_if_ratio_scored=[round((D[b].scores["demand"] + scorecard.score(cost, fb.rent_ratio[b])) / 2, 3) for b in fb.index]
          )[["emirate", "call", "composite", "composite_if_ratio_scored", "rent_ratio", "cost_if_ratio_scored", "confidence"]]

A cell with no DLD, ADREC or listings rent takes its emirate's median (SHOP_RENT_WHY). When half or more of an area's women with a rent live in such cells, the area's women-weighted median IS that one emirate-wide number, not an observation of the place: the rent test is neutral (passes) and the call low confidence. Lounges: a site cell on the fallback scores rent neutral (0.5), low confidence. Nine of the ten Sharjah GROW areas sat at ratio exactly 1.000, and manama-ajman / mudayrah-ajman missed at 0.997, on this number alone (2026-10-11). Known edge: sharjah-sharjah (15% fallback) keeps its observed-rent call, though its median lands on the fallback value (1.000; its listings cells alone give 0.997). 



,emirate,call,composite,composite_if_ratio_scored,rent_ratio,cost_if_ratio_scored,confidence
lounge,,,,,,,
al-ain,Abu Dhabi,PROTECT,0.724,0.826,1.19,0.71,low
al-falah,Abu Dhabi,PROTECT,0.664,0.767,1.19,0.71,low


## 3. Cannibalisation what-if (display only)

Close each lounge, re-split: `absorbed` = the share of its served women its siblings would take over. Not scored.

In [5]:
cn = F[~F.not_scored][["emirate", "served_women", "absorbed_share"]].assign(absorbed_women=lambda x: x.served_women * x.absorbed_share)
cn["call"] = cn.index.map(lambda b: D[b].action)
cn.sort_values("absorbed_share", ascending=False).round(2)

,emirate,served_women,absorbed_share,absorbed_women,call
branch_id,,,,,
noya-plaza,Abu Dhabi,1385.72,0.30,419.15,HOLD
westyas,Abu Dhabi,3160.09,0.27,850.06,SHRINK
khalifa-city-a,Abu Dhabi,16679.26,0.20,3308.14,HOLD
mohammed-bin-zayed-city,Abu Dhabi,4056.92,0.19,784.20,HOLD
shahama,Abu Dhabi,9865.31,0.19,1879.86,HOLD
shakhbout-city,Abu Dhabi,12743.27,0.19,2381.51,HOLD
al-maqta,Abu Dhabi,6989.73,0.18,1236.37,HOLD
ministries-complex,Abu Dhabi,8397.89,0.16,1370.32,HOLD
al-falah,Abu Dhabi,9509.65,0.15,1447.75,PROTECT


### Performance vs site (context, not scored)

`performance_vs_site` = the lounge's own served women ÷ what a typical lounge would serve at its site. Above 1, the lounge sells more than a typical lounge would at its site (measured by its reviews). Shown, not scored.

In [6]:
pv = F[~F.not_scored][["emirate", "review_count", "served_women", "site_served", "performance_vs_site"]].assign(call=lambda x: x.index.map(lambda b: D[b].action))
print("Spearman performance_vs_site vs reviews:", round(pv.performance_vs_site.corr(pv.review_count, method="spearman"), 2))
pv.sort_values("performance_vs_site", ascending=False).round(2)

Spearman performance_vs_site vs reviews: 0.95


,emirate,review_count,served_women,site_served,performance_vs_site,call
branch_id,,,,,,
khalifa-city-a,Abu Dhabi,2465,16679.26,6144.21,2.71,HOLD
khaleej-al-arabi,Abu Dhabi,1753,3083.19,1358.59,2.27,HOLD
ministries-complex,Abu Dhabi,1217,8397.89,3919.20,2.14,HOLD
al-ain,Abu Dhabi,1332,18651.84,10173.46,1.83,PROTECT
al-maqta,Abu Dhabi,1295,6989.73,4159.11,1.68,HOLD
shakhbout-city,Abu Dhabi,1277,12743.27,8685.03,1.47,HOLD
al-falah,Abu Dhabi,951,9509.65,7865.50,1.21,PROTECT
city-walk,Dubai,857,2155.24,1905.03,1.13,SHRINK
shahama,Abu Dhabi,757,9865.31,9865.31,1.00,HOLD


## 4. Composites against the lines

In [7]:
print(t.composite.describe().round(3))
print("within 0.05 of a line:", list(t.index[(abs(t.composite - scorecard.PROTECT_AT) < 0.05) | (abs(t.composite - scorecard.SHRINK_AT) < 0.05)]))
print("no rent ratio (cost scores neutral 0.5):", list(F.index[~F.not_scored & F.rent_ratio.isna()]))
print("confidence:", t.confidence.value_counts().to_dict())
dem = next(s for s in scorecard.SIGNALS if s.name == "demand")
print("with cost neutral (0.5), PROTECT needs demand_pct >=", round((2 * scorecard.PROTECT_AT - 0.5) * dem.best, 2),
      "and SHRINK needs demand_pct <=", round((2 * scorecard.SHRINK_AT - 0.5) * dem.best, 2))
print("cost score at rent_ratio 1:", round(scorecard.score(next(s for s in scorecard.SIGNALS if s.name == "cost"), 1.0), 3))

count    23.000
mean      0.513
std       0.187
min       0.151
25%       0.370
50%       0.496
75%       0.636
max       0.950
Name: composite, dtype: float64
within 0.05 of a line: ['al-falah', 'mohammed-bin-zayed-city', 'al-barsha', 'noya-plaza', 'delma', 'khalifa-city-a', 'ministries-complex', 'khaleej-al-arabi', 'westyas', 'al-jada']
no rent ratio (cost scores neutral 0.5): []
confidence: {'low': 12, 'high': 8, 'medium': 3}
with cost neutral (0.5), PROTECT needs demand_pct >= 0.8 and SHRINK needs demand_pct <= 0.2
cost score at rent_ratio 1: 0.471


## Open questions for review

1. **Every lounge is now judged against its emirate's salon sites, and four emirates don't have enough of them.** Dubai, Abu Dhabi and Sharjah rank against their own 582, 304 and 143 reference sites. Ajman (38), Ras Al Khaimah (33), Fujairah (18) and Umm Al Quwain (5) fall back to all 1,123 UAE sites, so ras-al-khaimah (86th percentile) and al-taif-mall (90th) are ranked mostly against Dubai and Abu Dhabi sites. Rent still uses each emirate's own median, so al-taif-mall's rent ratio of 1.55 rests on 18 Fujairah sites. All 23 lounges have a rent ratio, but al-ain and al-falah sit in cells on the Abu Dhabi emirate-median fallback (no ADREC rate for the district), so their rent scores a neutral 0.5 (question 2). If a rent did go missing, PROTECT would need `demand_pct` ≥ 0.80 and SHRINK ≤ 0.20. The median lounge sits at `demand_pct` 0.66 and `rent_ratio` 0.90.
2. **The calls: 6 PROTECT, 14 HOLD, 3 SHRINK, 1 not scored** (zayed-international-airport serves travellers). PROTECT: al-taif-mall, mirdif-35, al-dhafra, al-ain, nad-al-sheba, al-falah. SHRINK: city-walk, al-jada, westyas, one each in Dubai, Sharjah and Abu Dhabi. 10 of 14 HOLDs are in Abu Dhabi. **Fallback rents** (section 2, second table): al-ain and al-falah would score 0.83 and 0.77 if their fallback ratio of 1.19 counted; scored neutral they are 0.72 and 0.66, still PROTECT, low confidence.
3. **Ranking within the emirate means "good for Dubai" and "good for Abu Dhabi" are different bars.** A typical lounge at a Dubai site serves 1.9k–5.4k women, but Dubai's median reference site serves only 2,328. That makes mirdif-35 the 98th percentile in Dubai (75th UAE-wide), and nad-al-sheba PROTECT at the 68th percentile in Dubai (48th UAE-wide) on a site serving 2,670 women. Abu Dhabi's sites serve more, but its rents are dearer against its own median: khalifa-city-a is at the 73rd percentile on demand (78th UAE-wide), yet its rent ratio of 0.62 scores cost 0.00, which leaves it HOLD at 0.363. A PROTECT in one emirate doesn't compare with a PROTECT in another; the UAE-wide percentiles are the only common yardstick.
4. **Rent decides a lot, on a narrow scale, and the weakest rents drive it.** Cost scores 0 at a ratio of 0.62 or below and 1 at 1.43 or above, and the median salon site (ratio 1) scores 0.47. al-jada is SHRINK (0.338) despite the 66th percentile in Sharjah (85th UAE-wide): its listing asking rent (AED 1,771/m²) is 0.63 of Sharjah's median, so it scores cost 0.02. ras-al-khaimah (86th percentile) is HOLD at 0.496 because its listing rent of 1,172 is 0.73 of RAK's 853. city-walk is SHRINK at 0.151 on DLD rent of 4,561 (ratio 0.47, dearer than 98% of UAE reference sites) and Dubai's 30th percentile. al-taif-mall tops the list (0.950) on the cheapest rent in the whole reference set. al-ain and al-falah use the Abu Dhabi fallback median (1,168, ratio 1.19), not an observed rent, so it scores neutral.
5. **Scoring the site, not the lounge, ignores strong performers** (the performance-vs-site table after section 3). khalifa-city-a is HOLD although the model estimates this lounge is performing 2.7x compared to its site; khaleej-al-arabi (2.3x) and ministries-complex (2.1x) are HOLD only by 0.01 or less. At the other end, noya-plaza performs at 0.3x its site and is HOLD. Performance vs site tracks review count (Spearman 0.95), which is why it was left out of the score.
6. **Most calls sit near a line (the knife edge, docs/sanity-checks.md F7).** The median composite is 0.496: a percentile puts the median salon site at 0.5 and a rent ratio of 1 scores 0.47, so composites bunch in the middle. 10 lounges are within 0.05 of a line (al-falah, mohammed-bin-zayed-city, al-barsha, noya-plaza, delma, khalifa-city-a, ministries-complex, khaleej-al-arabi, westyas, al-jada), and 11 are low confidence (those 10 plus al-ain, on a fallback rent). Six calls flip with a ±25% weight: khalifa-city-a, ministries-complex and khaleej-al-arabi to SHRINK, westyas and al-jada to HOLD, and al-falah to HOLD. khaleej-al-arabi sits 0.0004 above SHRINK. Only city-walk is a firm SHRINK.
7. **Keep and grow use different bars** (docs/sanity-checks.md F6). nad-al-sheba is PROTECT at the 68th percentile of Dubai sites, yet a typical lounge there serves 2,670 women, under growth's absolute 6,000 net-new line: the same site would be SKIP as a new opening (`growth.ipynb`).
8. **Cannibalisation is display only.** It is highest at noya-plaza (0.30) and westyas (0.27), 0.01 at every Dubai lounge, and 0.00 at al-ain, al-dhafra, ras-al-khaimah and al-taif-mall. Is showing it without scoring it enough?